## Unilingual Model Exploration

This section explores unilingual models (Ensemble methods) that uses one model per language


---
Note that cross-validation process differs if we use a multi-lingual model or mono-lingual model:
- Multi-Lingual: Each fold should contain all the nodes with the same sentence_id and for all languages! (To avoid unbalance)
- Uni-Lingual: Each fold should contain all the the nodes with the same sentence_id. There are 2 ways to do this:

In [1]:
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import *
from src.submission import generate_kaggle_submission


from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 



train = pd.read_csv("./data/train_dataset_processed.csv")
test = pd.read_csv("./data/test_dataset_processed.csv")
train

,sentence_id,language,node,length,degree,avg_neighbor_deg,degree_squared,degree_diff,clustering,local_degree_ratio,max_neighbor_degree,degree_centrality,harmonic_centrality,betweenness_centrality,pagerank,root
0,2,Japanese,14,23,2,2.000000,4,0.000000,0,0.999995,2,0.090909,5.865512,0.173160,0.046568,0
1,2,Japanese,8,23,2,2.000000,4,0.000000,0,0.999995,2,0.090909,6.382179,0.246753,0.044352,0
2,2,Japanese,4,23,1,2.000000,1,-1.000000,0,0.499998,2,0.045455,4.561122,0.000000,0.027162,0
3,2,Japanese,6,23,2,2.000000,4,0.000000,0,0.999995,3,0.090909,5.823846,0.090909,0.048565,0
4,2,Japanese,2,23,3,1.666667,9,1.333333,0,1.799989,2,0.136364,6.991703,0.255411,0.066901,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197474,995,Russian,2,19,1,3.000000,1,-2.000000,0,0.333332,3,0.055556,5.302381,0.000000,0.030321,0
197475,995,Russian,14,19,1,5.000000,1,-4.000000,0,0.200000,5,0.055556,6.034524,0.000000,0.029739,0
197476,995,Russian,5,19,2,3.000000,4,-1.000000,0,0.666664,5,0.111111,6.701190,0.111111,0.057065,0
197477,995,Russian,16,19,1,2.000000,1,-1.000000,0,0.499998,2,0.055556,5.005159,0.000000,0.032147,0


### Model 1: Random Forest Ensemble

UnilingualEnsembleClassifier(base_model_kwargs={}, n_jobs=21)

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=RandomForestClassifier,
    base_model_kwargs={'n_estimators': 100, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'n_estimators': [250, 500],            # Number of trees
    'max_depth': [None, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced']         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

UnilingualEnsembleClassifier(base_model_kwargs={'n_estimators': 100,
                                                'n_jobs': 1},
                             cv=2, gridsearch_per_language=True, n_jobs=21,
                             param_grid={'class_weight': [None, 'balanced'],
                                         'max_depth': [None, 10, 20],
                                         'n_estimators': [250, 500]})

### Send this shit to kaggle

In [1]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()

# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,
    X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


NameError: name 'test' is not defined

## Model 2: XGBoost Ensemble

In [ ]:
# Imports
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import run_groupkfold_cv
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
train = pd.read_csv("./data/train_dataset_processed.csv")
test = pd.read_csv("./data/test_dataset_processed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset

# Run GroupKFold CV on train
cv_scores = run_groupkfold_cv(
    X=train,
    y=train[target_col],
    group_colname=group_col,
    clf_cls=UnilingualEnsembleClassifier,
    clf_kwargs={
        'base_model_cls': XGBClassifier,
        'base_model_kwargs': {'n_estimators': 300, 'n_jobs': 1},
        'language_colname': 'language',
        'n_jobs': 8  # adjust based on your CPU
    },
    n_splits=5,
    metric_fn=accuracy_score,
    verbose=True
)

# Plot CV scores
plt.plot(cv_scores, marker='o')
plt.title('CV Accuracy Scores per Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=XGBClassifier,
    base_model_kwargs={'n_estimators': 300, 'n_jobs': 1},
    language_colname='language',
    n_jobs=8
)
model.fit(train.drop(columns=target_col), train[target_col])

# Predict on test
test_preds = model.predict(test)

# If test has true labels, evaluate; otherwise just show predictions count
if target_col in test.columns:
    test_acc = accuracy_score(test[target_col], test_preds)
    print(f"Test Accuracy: {test_acc:.4f}")
else:
    print(f"Predicted {len(test_preds)} test samples.")

# Show a few predictions
print("Sample predictions:", test_preds[:10])


AttributeError: 'numpy.ndarray' object has no attribute 'groupby'

In [5]:
# === Prepare Kaggle Submission === #

# Attach predictions to test DataFrame
test = test.copy()
test["root"] = test_preds  # 0 or 1

# Safety check: make sure 'node', 'language', and 'sentence_id' exist
required_cols = {'node', 'language', 'sentence_id'}
if not required_cols.issubset(test.columns):
    missing = required_cols - set(test.columns)
    raise ValueError(f"Missing required columns in test set: {missing}")

# Function to find root node per sentence group
def find_root(group):
    root_rows = group[group["root"] == 1]
    if not root_rows.empty:
        return root_rows.iloc[0]["node"]
    else:
        return 1  # fallback value if no root was predicted

# Group by language and sentence_id, apply root selector
roots = test.groupby(["language", "sentence_id"]).apply(find_root).reset_index(name="root")

# Add sequential 'id' column
roots.insert(0, "id", range(1, len(roots) + 1))

# Save submission file
submission = roots[["id", "root"]]
submission.to_csv("data/unilingual_xgb.csv", index=False)

print("Submission file saved to: data/unilingual_xgb.csv")
display(submission.head())


Submission file saved to: data/unilingual_xgb.csv


/tmp/ipykernel_111973/3078533793.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  roots = test.groupby(["language", "sentence_id"]).apply(find_root).reset_index(name="root")


,id,root
0,1,13
1,2,14
2,3,1
3,4,1
4,5,1


## Model 3: LightGBM Ensemble

In [2]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=LGBMClassifier,
    base_model_kwargs={'n_estimators': 500, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'max_depth': [None, 5, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced'],         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

[LightGBM] [Info] Number of positive: 250, number of negative: 3143
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001144 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Number of positive: 250, number of negative: 5345
[LightGBM] [Info] Total Bins 4236
[LightGBM] [Info] Number of data points in the train set: 3393, number of used features: 25
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000413 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4694
[LightGBM] [Info] Number of data points in the train set: 5595, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.073681 -> initscore=-2.531472
[LightGBM] [Info] Start training from score -2.531472
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.044683 -> initscore=-3.062456
[LightGBM]

UnilingualEnsembleClassifier(base_model_cls=<class 'lightgbm.sklearn.LGBMClassifier'>,
                             base_model_kwargs={'n_estimators': 500,
                                                'n_jobs': 1},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'class_weight': [None, 'balanced'],
                                         'max_depth': [None, 5, 10, 20]})

In [ ]:
from src.submission import generate_kaggle_submission
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()

# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,
    X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_lightgbm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 12:06:13.179 | INFO     | src.submission:generate_kaggle_submission:28 - Generating predictions...
2025-05-29 12:06:22.613 | SUCCESS  | src.submission:generate_kaggle_submission:75 - Submission saved to: data/predictions_submission_unilingual_lightgbm.csv


## Model 4: CatBoost Ensemble (ligth)